# ⚡ Suha AGI Studio — 100% Free GPU Inference Bridge
Jambatan inferens Google Colab GPU (Tesla T4 16GB VRAM) untuk model Uncensored dari Hugging Face ke 9Router VPS.

In [ ]:
# 1. Pasang enjin inferens berkelajuan tinggi llama-cpp-python & Cloudflare Tunnel
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python uvicorn fastapi huggingface_hub -q
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -o cloudflared.deb
!dpkg -i cloudflared.deb > /dev/null
print('✓ CUDA Inference Engine & Cloudflared Ready!')

In [ ]:
# 2. Muat turun model Qwen Instruct GGUF daripada Hugging Face
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id='bartowski/Qwen2.5-7B-Instruct-1M-GGUF',
    filename='Qwen2.5-7B-Instruct-1M-Q4_K_M.gguf'
)
print(f'✓ Model Loaded at: {model_path}')

In [ ]:
# 3. Lancarkan Pelayan OpenAI API Serasi (Port 8000) dengan 35 Layers Offloaded ke GPU VRAM
import subprocess, time

cmd = f'python3 -m llama_cpp.server --model {model_path} --n_gpu_layers 35 --n_ctx 8192 --port 8000 --host 0.0.0.0'
proc = subprocess.Popen(cmd, shell=True)
time.sleep(5)
print('🟢 OpenAI-Compatible GPU Server LIVE on Port 8000!')

In [ ]:
# 4. Buka Terowong Percuma Cloudflare Quick Tunnel (Sifar Daftar, Sifar Token)
import subprocess, re, time

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True
)

public_url = None
for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print('='*60)
    print(f'⚡ LIVE INFERENCE ENDPOINT: {public_url}')
    print('='*60)
else:
    print('Sila jalankan semula sel ini.')